# IDRiD Microaneurysms (MA) — five-model comparison

This notebook trains and evaluates **U-Net, Attention U-Net, U-Net++, DeepLabV3+, and MSAR-UNet** for Microaneurysms (MA). It is self-contained: it does not import `segmentation_common.py` or any other project Python file.

**Dataset root expected:** `/content/drive/MyDrive/Research/A. Segmentation`

The official IDRiD testing split is kept separate. Only images with an existing lesion annotation mask are used for that lesion; a missing mask is **not** treated as a negative mask. Run the notebook top-to-bottom. GPU runtime is recommended.


In [1]:
# 1. Mount Drive and configure this experiment
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, random, time, copy, json, math
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

DATA_ROOT = Path('/content/drive/MyDrive/Research/A. Segmentation')
IMAGE_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 30                 # For a quick smoke test, change to 1 or 2 first
LEARNING_RATE = 1e-4
SEED = 42
NUM_WORKERS = 2
THRESHOLD = 0.5              # Fixed threshold; do not tune on the test set

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert DATA_ROOT.exists(), f'Dataset folder not found: {DATA_ROOT}'


ValueError: mount failed

In [ ]:
# 2. Find the official image and ground-truth folders and pair images with masks
LESION = 'MA'
LESION_FOLDER = '1. Microaneurysms'
MASK_SUFFIX = 'MA'

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

def find_named_dir(root: Path, target: str):
    target_norm = target.strip().lower()
    candidates = [p for p in root.rglob('*') if p.is_dir() and p.name.strip().lower() == target_norm]
    if not candidates:
        # permit punctuation/case variations
        candidates = [p for p in root.rglob('*') if p.is_dir() and target_norm in p.name.strip().lower()]
    if not candidates:
        raise FileNotFoundError(f'Cannot find folder named like {target!r} under {root}')
    return candidates[0]

def find_split_dir(parent: Path, split_name: str):
    target = split_name.lower()
    candidates = [p for p in parent.iterdir() if p.is_dir() and target in p.name.lower()]
    if not candidates:
        raise FileNotFoundError(f'Cannot find {split_name!r} inside {parent}')
    return candidates[0]

def get_id(path: Path):
    match = re.search(r'IDRiD[_-]?(\d+)', path.stem, flags=re.IGNORECASE)
    return int(match.group(1)) if match else None

def files_by_id(folder: Path, mask=False):
    found = {}
    duplicates = []
    for p in sorted(folder.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXTS:
            continue
        if mask and not re.search(r'_' + re.escape(MASK_SUFFIX) + r'(?:\D|$)', p.stem, flags=re.IGNORECASE):
            continue
        image_id = get_id(p)
        if image_id is None:
            continue
        if image_id in found:
            duplicates.append((image_id, str(found[image_id]), str(p)))
            # Prefer TIFF masks when duplicate export variants exist.
            if p.suffix.lower() in {'.tif', '.tiff'} and found[image_id].suffix.lower() not in {'.tif', '.tiff'}:
                found[image_id] = p
        else:
            found[image_id] = p
    if duplicates:
        print(f'Warning: {len(duplicates)} duplicate file(s) found; one file per ID will be used.')
    return found

images_root = find_named_dir(DATA_ROOT, '1. Original Images')
gt_root = find_named_dir(DATA_ROOT, '2. All Segmentation Groundtruths')
train_images_dir = find_split_dir(images_root, 'Training Set')
test_images_dir = find_split_dir(images_root, 'Testing Set')
train_gt_dir = find_split_dir(gt_root, 'Training Set')
test_gt_dir = find_split_dir(gt_root, 'Testing Set')
train_mask_dir = find_named_dir(train_gt_dir, LESION_FOLDER)
test_mask_dir = find_named_dir(test_gt_dir, LESION_FOLDER)

train_images = files_by_id(train_images_dir)
test_images = files_by_id(test_images_dir)
train_masks = files_by_id(train_mask_dir, mask=True)
test_masks = files_by_id(test_mask_dir, mask=True)

def pair_records(images, masks):
    rows = []
    for image_id, image_path in sorted(images.items()):
        mask_path = masks.get(image_id)
        if mask_path is not None:
            rows.append({'id': image_id, 'image': str(image_path), 'mask': str(mask_path)})
    return rows

train_records = pair_records(train_images, train_masks)
test_records = pair_records(test_images, test_masks)

print('Lesion:', LESION)
print('Official training images:', len(train_images), '| masks:', len(train_masks), '| matched pairs:', len(train_records))
print('Official testing images:', len(test_images), '| masks:', len(test_masks), '| matched pairs:', len(test_records))
print('Training image IDs without annotation (excluded, not labelled negative):', sorted(set(train_images)-set(train_masks)))
print('Testing image IDs without annotation (excluded, not labelled negative):', sorted(set(test_images)-set(test_masks)))
assert len(train_records) >= 5, 'Too few training image/mask pairs; check folder paths and filenames.'
assert len(test_records) >= 1, 'No official test image/mask pairs found; check folder paths and filenames.'


In [ ]:
# 3. Self-contained PyTorch model architectures
# Every model returns logits shaped [batch, 1, height, width].
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.skip = nn.Identity() if in_ch == out_ch else nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch))
    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)), inplace=True)
        y = self.bn2(self.conv2(y))
        return F.relu(y + self.skip(x), inplace=True)

def up_to(x, reference):
    return F.interpolate(x, size=reference.shape[-2:], mode='bilinear', align_corners=False)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.e0=ConvBlock(3,32); self.e1=ConvBlock(32,64); self.e2=ConvBlock(64,128); self.e3=ConvBlock(128,256); self.mid=ConvBlock(256,512)
        self.d3=ConvBlock(512+256,256); self.d2=ConvBlock(256+128,128); self.d1=ConvBlock(128+64,64); self.d0=ConvBlock(64+32,32)
        self.out=nn.Conv2d(32,1,1)
    def forward(self,x):
        x0=self.e0(x); x1=self.e1(self.pool(x0)); x2=self.e2(self.pool(x1)); x3=self.e3(self.pool(x2)); m=self.mid(self.pool(x3))
        y=self.d3(torch.cat([up_to(m,x3),x3],1)); y=self.d2(torch.cat([up_to(y,x2),x2],1)); y=self.d1(torch.cat([up_to(y,x1),x1],1)); y=self.d0(torch.cat([up_to(y,x0),x0],1))
        return self.out(y)

class AttentionGate(nn.Module):
    def __init__(self, gate_ch, skip_ch, inter_ch):
        super().__init__()
        self.g = nn.Sequential(nn.Conv2d(gate_ch,inter_ch,1,bias=False),nn.BatchNorm2d(inter_ch))
        self.x = nn.Sequential(nn.Conv2d(skip_ch,inter_ch,1,bias=False),nn.BatchNorm2d(inter_ch))
        self.psi = nn.Sequential(nn.ReLU(inplace=True),nn.Conv2d(inter_ch,1,1),nn.Sigmoid())
    def forward(self,g,x):
        a=self.psi(self.g(up_to(g,x))+self.x(x))
        return x*a

class AttentionUNet(nn.Module):
    def __init__(self):
        super().__init__(); self.pool=nn.MaxPool2d(2)
        self.e0=ConvBlock(3,32); self.e1=ConvBlock(32,64); self.e2=ConvBlock(64,128); self.e3=ConvBlock(128,256); self.mid=ConvBlock(256,512)
        self.a3=AttentionGate(512,256,128); self.a2=AttentionGate(256,128,64); self.a1=AttentionGate(128,64,32); self.a0=AttentionGate(64,32,16)
        self.d3=ConvBlock(768,256); self.d2=ConvBlock(384,128); self.d1=ConvBlock(192,64); self.d0=ConvBlock(96,32); self.out=nn.Conv2d(32,1,1)
    def forward(self,x):
        x0=self.e0(x); x1=self.e1(self.pool(x0)); x2=self.e2(self.pool(x1)); x3=self.e3(self.pool(x2)); m=self.mid(self.pool(x3))
        y=self.d3(torch.cat([up_to(m,x3),self.a3(m,x3)],1)); y=self.d2(torch.cat([up_to(y,x2),self.a2(y,x2)],1)); y=self.d1(torch.cat([up_to(y,x1),self.a1(y,x1)],1)); y=self.d0(torch.cat([up_to(y,x0),self.a0(y,x0)],1))
        return self.out(y)

class UNetPlusPlus(nn.Module):
    """Nested dense skip connections; compact four-level U-Net++ with deep supervision disabled."""
    def __init__(self):
        super().__init__(); self.pool=nn.MaxPool2d(2)
        self.x00=ConvBlock(3,32); self.x10=ConvBlock(32,64); self.x20=ConvBlock(64,128); self.x30=ConvBlock(128,256)
        self.x01=ConvBlock(32+64,32); self.x11=ConvBlock(64+128,64); self.x21=ConvBlock(128+256,128)
        self.x02=ConvBlock(32*2+64,32); self.x12=ConvBlock(64*2+128,64)
        self.x03=ConvBlock(32*3+64,32)
        self.out=nn.Conv2d(32,1,1)
    def forward(self,x):
        x00=self.x00(x); x10=self.x10(self.pool(x00)); x20=self.x20(self.pool(x10)); x30=self.x30(self.pool(x20))
        x01=self.x01(torch.cat([x00,up_to(x10,x00)],1))
        x11=self.x11(torch.cat([x10,up_to(x20,x10)],1))
        x21=self.x21(torch.cat([x20,up_to(x30,x20)],1))
        x02=self.x02(torch.cat([x00,x01,up_to(x11,x00)],1))
        x12=self.x12(torch.cat([x10,x11,up_to(x21,x10)],1))
        x03=self.x03(torch.cat([x00,x01,x02,up_to(x12,x00)],1))
        return self.out(x03)

class ASPP(nn.Module):
    def __init__(self,in_ch,out_ch=128):
        super().__init__()
        self.branches=nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(inplace=True)),
            nn.Sequential(nn.Conv2d(in_ch,out_ch,3,padding=3,dilation=3,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(inplace=True)),
            nn.Sequential(nn.Conv2d(in_ch,out_ch,3,padding=6,dilation=6,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(inplace=True)),
            nn.Sequential(nn.Conv2d(in_ch,out_ch,3,padding=9,dilation=9,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(inplace=True)),
        ])
        self.image_pool=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(in_ch,out_ch,1,bias=False),nn.ReLU(inplace=True))
        self.project=nn.Sequential(nn.Conv2d(out_ch*5,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(inplace=True),nn.Dropout2d(.1))
    def forward(self,x):
        pooled=F.interpolate(self.image_pool(x),size=x.shape[-2:],mode='bilinear',align_corners=False)
        return self.project(torch.cat([*(b(x) for b in self.branches),pooled],1))

class DeepLabV3Plus(nn.Module):
    """DeepLabV3+-style encoder, ASPP context module and low-level feature decoder."""
    def __init__(self):
        super().__init__()
        self.stem=ConvBlock(3,32)
        self.low=ConvBlock(32,64); self.mid=ConvBlock(64,128); self.high=ConvBlock(128,256)
        self.pool=nn.MaxPool2d(2); self.aspp=ASPP(256,128)
        self.low_project=nn.Sequential(nn.Conv2d(64,48,1,bias=False),nn.BatchNorm2d(48),nn.ReLU(inplace=True))
        self.decoder=nn.Sequential(ConvBlock(176,128),nn.Conv2d(128,64,3,padding=1,bias=False),nn.BatchNorm2d(64),nn.ReLU(inplace=True),nn.Conv2d(64,1,1))
    def forward(self,x):
        x0=self.stem(x); low=self.low(self.pool(x0)); mid=self.mid(self.pool(low)); high=self.high(self.pool(mid))
        context=self.aspp(high); context=up_to(context,low); low=self.low_project(low)
        return F.interpolate(self.decoder(torch.cat([context,low],1)),size=x.shape[-2:],mode='bilinear',align_corners=False)

class MultiScaleBlock(nn.Module):
    def __init__(self,in_ch,out_ch):
        super().__init__()
        branch=max(out_ch//4,8)
        self.b1=nn.Sequential(nn.Conv2d(in_ch,branch,1,bias=False),nn.BatchNorm2d(branch),nn.ReLU(inplace=True))
        self.b3=nn.Sequential(nn.Conv2d(in_ch,branch,3,padding=1,bias=False),nn.BatchNorm2d(branch),nn.ReLU(inplace=True))
        self.b5=nn.Sequential(nn.Conv2d(in_ch,branch,5,padding=2,bias=False),nn.BatchNorm2d(branch),nn.ReLU(inplace=True))
        self.bd=nn.Sequential(nn.Conv2d(in_ch,branch,3,padding=2,dilation=2,bias=False),nn.BatchNorm2d(branch),nn.ReLU(inplace=True))
        self.fuse=nn.Sequential(nn.Conv2d(branch*4,out_ch,1,bias=False),nn.BatchNorm2d(out_ch))
        self.skip=nn.Identity() if in_ch==out_ch else nn.Conv2d(in_ch,out_ch,1,bias=False)
    def forward(self,x):
        z=self.fuse(torch.cat([self.b1(x),self.b3(x),self.b5(x),self.bd(x)],1))
        return F.relu(z+self.skip(x),inplace=True)

class MSARUNet(nn.Module):
    """Proposed Multi-Scale Attention Residual U-Net: multiscale residual blocks + attention skip gates."""
    def __init__(self):
        super().__init__(); self.pool=nn.MaxPool2d(2)
        self.e0=MultiScaleBlock(3,32); self.e1=ResidualBlock(32,64); self.e2=MultiScaleBlock(64,128); self.e3=ResidualBlock(128,256); self.mid=MultiScaleBlock(256,512)
        self.a3=AttentionGate(512,256,128); self.a2=AttentionGate(256,128,64); self.a1=AttentionGate(128,64,32); self.a0=AttentionGate(64,32,16)
        self.d3=ResidualBlock(768,256); self.d2=ResidualBlock(384,128); self.d1=ResidualBlock(192,64); self.d0=ResidualBlock(96,32); self.out=nn.Conv2d(32,1,1)
    def forward(self,x):
        x0=self.e0(x); x1=self.e1(self.pool(x0)); x2=self.e2(self.pool(x1)); x3=self.e3(self.pool(x2)); m=self.mid(self.pool(x3))
        y=self.d3(torch.cat([up_to(m,x3),self.a3(m,x3)],1)); y=self.d2(torch.cat([up_to(y,x2),self.a2(y,x2)],1)); y=self.d1(torch.cat([up_to(y,x1),self.a1(y,x1)],1)); y=self.d0(torch.cat([up_to(y,x0),self.a0(y,x0)],1))
        return self.out(y)

MODEL_NAMES = ['UNet', 'AttentionUNet', 'UNetPlusPlus', 'DeepLabV3Plus', 'MSARUNet']
def make_model(name):
    constructors = {'UNet':UNet, 'AttentionUNet':AttentionUNet, 'UNetPlusPlus':UNetPlusPlus, 'DeepLabV3Plus':DeepLabV3Plus, 'MSARUNet':MSARUNet}
    if name not in constructors:
        raise ValueError(f'Unknown model {name}. Choose from {list(constructors)}')
    return constructors[name]()

# Quick architecture-only output shape check; no training occurs here.
with torch.no_grad():
    sample = torch.randn(1,3,IMAGE_SIZE,IMAGE_SIZE).to(DEVICE)
    for model_name in MODEL_NAMES:
        model = make_model(model_name).to(DEVICE).eval()
        result = model(sample)
        print(model_name, '->', tuple(result.shape))
        assert tuple(result.shape) == (1,1,IMAGE_SIZE,IMAGE_SIZE)
        del model
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()


In [ ]:
# 4. Image and mask preprocessing; train/validation/test datasets

def load_rgb(path):
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f'Could not read image: {path}')
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
    # Apply CLAHE to luminance (LAB L channel), not to a chromatic channel.
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(l)
    image = cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2RGB)
    return image.astype(np.float32) / 255.0

def load_binary_mask(path):
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f'Could not read mask: {path}')
    mask = cv2.resize(mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)
    return (mask > 0).astype(np.float32)

class LesionDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records = records
        self.augment = augment
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        row = self.records[idx]
        image = load_rgb(row['image'])
        mask = load_binary_mask(row['mask'])
        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=1).copy(); mask = np.flip(mask, axis=1).copy()
            if random.random() < 0.5:
                image = np.flip(image, axis=0).copy(); mask = np.flip(mask, axis=0).copy()
        image = torch.from_numpy(image.transpose(2,0,1).copy()).float()
        mask = torch.from_numpy(mask[None,:,:].copy()).float()
        return image, mask

train_rows, val_rows = train_test_split(train_records, test_size=0.2, random_state=SEED)
train_ds = LesionDataset(train_rows, augment=True)
val_ds = LesionDataset(val_rows, augment=False)
test_ds = LesionDataset(test_records, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
print(f'Train={len(train_ds)}, validation={len(val_ds)}, official test={len(test_ds)}')


In [ ]:
# 5. Training, validation selection, metrics, and saved predictions
RESULT_DIR = Path('/content/drive/MyDrive/Research/segmentation_results') / LESION
RESULT_DIR.mkdir(parents=True, exist_ok=True)


def dice_loss_logits(logits, targets, smooth=1e-6):
    probs = torch.sigmoid(logits)
    probs = probs.flatten(1); targets = targets.flatten(1)
    intersection = (probs * targets).sum(1)
    return (1 - ((2*intersection + smooth) / (probs.sum(1) + targets.sum(1) + smooth))).mean()

def combined_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets) + dice_loss_logits(logits, targets)

@torch.no_grad()
def evaluate_loss_and_dice(model, loader):
    model.eval(); losses=[]; dices=[]
    for images, masks in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        logits = model(images)
        losses.append(combined_loss(logits, masks).item())
        pred = (torch.sigmoid(logits) >= THRESHOLD).float()
        inter = (pred*masks).sum(dim=(1,2,3))
        denom = pred.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3))
        dices.extend(((2*inter+1e-6)/(denom+1e-6)).cpu().tolist())
    return float(np.mean(losses)), float(np.mean(dices))

@torch.no_grad()
def evaluate_test(model, loader, model_name):
    model.eval(); tp=fp=fn=tn=0; infer_seconds=0.0; n_images=0
    for images, masks in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        start=time.perf_counter(); logits=model(images)
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        infer_seconds += time.perf_counter()-start; n_images += images.shape[0]
        pred=(torch.sigmoid(logits)>=THRESHOLD)
        target=(masks>=0.5)
        tp += int((pred & target).sum().item()); fp += int((pred & ~target).sum().item())
        fn += int((~pred & target).sum().item()); tn += int((~pred & ~target).sum().item())
    eps=1e-8
    precision=tp/(tp+fp+eps); recall=tp/(tp+fn+eps)
    dice=(2*tp)/(2*tp+fp+fn+eps); iou=tp/(tp+fp+fn+eps)
    specificity=tn/(tn+fp+eps); pixel_accuracy=(tp+tn)/(tp+tn+fp+fn+eps)
    return {'lesion':LESION,'model':model_name,'test_dice':dice,'test_iou':iou,'precision':precision,'recall_sensitivity':recall,'specificity':specificity,'pixel_accuracy':pixel_accuracy,'inference_ms_per_image':1000*infer_seconds/max(n_images,1),'TP':tp,'FP':fp,'FN':fn,'TN':tn,'threshold':THRESHOLD,'test_images':len(test_records)}

def fit_model(model_name):
    print('\n' + '='*72 + '\nTraining:', model_name, '| lesion:', LESION)
    model=make_model(model_name).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    best_val_dice=-1.0; best_epoch=0; best_path=RESULT_DIR/f'{model_name}_best.pth'; history=[]
    start_all=time.perf_counter()
    for epoch in range(1,EPOCHS+1):
        model.train(); train_losses=[]
        for images,masks in train_loader:
            images,masks=images.to(DEVICE),masks.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits=model(images); loss=combined_loss(logits,masks)
            loss.backward(); optimizer.step(); train_losses.append(loss.item())
        val_loss,val_dice=evaluate_loss_and_dice(model,val_loader)
        train_loss=float(np.mean(train_losses))
        history.append({'model':model_name,'epoch':epoch,'train_loss':train_loss,'val_loss':val_loss,'val_dice':val_dice})
        print(f'Epoch {epoch:02d}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_dice={val_dice:.4f}')
        if val_dice>best_val_dice:
            best_val_dice=val_dice; best_epoch=epoch
            torch.save({'model_name':model_name,'model_state_dict':model.state_dict(),'epoch':epoch,'val_dice':val_dice,'image_size':IMAGE_SIZE},best_path)
    train_seconds=time.perf_counter()-start_all
    pd.DataFrame(history).to_csv(RESULT_DIR/f'{model_name}_history.csv',index=False)
    checkpoint=torch.load(best_path,map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    metrics=evaluate_test(model,test_loader,model_name)
    metrics.update({'best_epoch':best_epoch,'best_val_dice':best_val_dice,'training_minutes':train_seconds/60,'parameters':sum(p.numel() for p in model.parameters())})
    # Save per-image probability overlays and binary predictions for visual inspection.
    pred_dir=RESULT_DIR/model_name/'predictions'; pred_dir.mkdir(parents=True,exist_ok=True)
    model.eval()
    with torch.no_grad():
        for row in test_records:
            rgb=load_rgb(row['image']); x=torch.from_numpy(rgb.transpose(2,0,1).copy()).unsqueeze(0).float().to(DEVICE)
            prob=torch.sigmoid(model(x))[0,0].cpu().numpy()
            binary=(prob>=THRESHOLD).astype(np.uint8)*255
            original=(rgb*255).clip(0,255).astype(np.uint8)
            overlay=original.copy(); overlay[binary>0]=((0.55*overlay[binary>0])+(0.45*np.array([255,0,0]))).astype(np.uint8)
            image_id=row['id']
            Image.fromarray(binary).save(pred_dir/f'IDRiD_{image_id:02d}_{LESION}_prediction.png')
            Image.fromarray(overlay).save(pred_dir/f'IDRiD_{image_id:02d}_{LESION}_overlay.png')
    del model
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return metrics

print('Results will be saved to:', RESULT_DIR)
print('Training configuration:', {'epochs':EPOCHS,'batch_size':BATCH_SIZE,'lr':LEARNING_RATE,'image_size':IMAGE_SIZE,'threshold':THRESHOLD})


In [ ]:
# 6. Run the five-model comparison
# Start with EPOCHS=1 or 2 to check that the full pipeline works. Then set EPOCHS to your planned value.
results=[]
for model_name in MODEL_NAMES:
    result=fit_model(model_name)
    results.append(result)
    pd.DataFrame(results).to_csv(RESULT_DIR/'comparison_results_partial.csv',index=False)

results_df=pd.DataFrame(results).sort_values('test_dice',ascending=False)
results_path=RESULT_DIR/'comparison_results.csv'
results_df.to_csv(results_path,index=False)
print('\nFinal comparison results:')
display(results_df)
print('Saved results to:', results_path)


In [ ]:
# 7. Plot comparison metrics and inspect a sample prediction
metrics_to_plot=['test_dice','test_iou','precision','recall_sensitivity']
plot_df=results_df.set_index('model')
for metric in metrics_to_plot:
    ax=plot_df[metric].sort_values(ascending=False).plot(kind='bar',figsize=(9,4),title=f'{LESION}: {metric.replace("_"," ")}')
    ax.set_ylim(0,1); ax.set_ylabel('Score'); ax.set_xlabel('Model'); plt.xticks(rotation=25,ha='right'); plt.tight_layout(); plt.show()

# Show a few ground-truth / prediction overlays from the best-performing model.
best_model_name=str(results_df.iloc[0]['model'])
best_pred_dir=RESULT_DIR/best_model_name/'predictions'
examples=test_records[:min(3,len(test_records))]
fig,axes=plt.subplots(len(examples),3,figsize=(12,4*len(examples)))
if len(examples)==1: axes=np.expand_dims(axes,0)
for i,row in enumerate(examples):
    img=np.array(Image.open(row['image']).convert('RGB').resize((IMAGE_SIZE,IMAGE_SIZE)))
    gt=load_binary_mask(row['mask'])
    pred_path=best_pred_dir/f"IDRiD_{row['id']:02d}_{LESION}_prediction.png"
    pred=np.array(Image.open(pred_path).convert('L'))>0
    axes[i,0].imshow(img); axes[i,0].set_title(f"Image ID {row['id']}")
    axes[i,1].imshow(gt,cmap='gray'); axes[i,1].set_title('Ground truth mask')
    axes[i,2].imshow(pred,cmap='gray'); axes[i,2].set_title(f'{best_model_name} prediction')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout(); plt.show()


## Research notes and reporting checklist

- Report the exact train/validation/test pair counts printed above.
- The official test set is used only for final evaluation; use validation Dice for checkpoint selection.
- Dice and IoU are computed from the global pixel counts on the test set. Pixel accuracy can look high when lesion pixels are rare, so do not use it alone.
- This notebook uses a fixed probability threshold of `0.5`. If you decide to tune a threshold, tune it on validation data only and document that choice.
- The proposed MSAR-UNet combines multi-scale convolution branches, residual blocks, and attention-gated skip connections. Establish novelty with a literature review and ablation experiments; the name alone does not prove novelty.
- Save your notebook outputs and record the runtime/GPU details for reproducibility.
